# 09 — Realistic lagged mirror  (does the exit edge survive latency?)
Notebook 08 showed that mirroring the sharps' **entries AND exits** is profitable at the
*frictionless* ceiling (~+11% per position, positive median, consensus helps). The whole
project has taught us one lesson though: **latency destroys copy edges.** This notebook is
the decisive stress-test.

**What it does:** for a sample of point-in-time-qualified positions, it reconstructs the
sharp's actual round trip from their trade activity (when they bought, when they sold), then
simulates a *realistic* copy:
- **Enter** `COPY_LAG_HOURS` after their buy, at the market price *then*, paying slippage.
- **Exit** `COPY_LAG_HOURS` after their sell, at the market price *then*, paying slippage —
  or, if they never sold, **hold to resolution** like they did.

The gap between this and the notebook-08 ceiling is the tax you pay for being a step behind.
If the lagged return is still clearly positive → we have a real, tradeable strategy worth
paper-trading. If latency eats it → the ceiling was a mirage and we've hit the wall.


In [1]:
import importlib, pmc, time, random
importlib.reload(pmc)
from pmc import CFG, get_leaderboard, get_closed_positions, get_user_trades, price_at
import pandas as pd, numpy as np
print(f"copy lag = {CFG.COPY_LAG_HOURS}h | sample = {CFG.MIRROR_SAMPLE} positions | "
      f"slippage = {CFG.SLIPPAGE_TOLERANCE}")

copy lag = 6h | sample = 600 positions | slippage = 0.06


## 1. Rebuild the point-in-time-qualified positions (cached, fast)
Identical to notebook 08: candidate pool → resolved history → PIT qualification → backer count.

In [2]:
cands = {}
for w in ("ALL", "MONTH"):
    for r in get_leaderboard(window=w, limit=CFG.WF_CANDIDATES):
        cands.setdefault(r["wallet"], r)
rows = []
for wallet in cands:
    for p in get_closed_positions(wallet, max_positions=600):
        cost = float(p.get("totalBought") or 0)
        if cost <= 0:
            continue
        rows.append({"wallet": wallet, "conditionId": p.get("conditionId"), "outcome": p.get("outcome"),
                     "asset": p.get("asset"), "entry": float(p.get("avgPrice") or 0),
                     "entry_ts": int(p.get("timestamp") or 0), "endDate": p.get("endDate"),
                     "realizedPnl": float(p.get("realizedPnl") or 0), "cost": cost,
                     "won": 1 if float(p.get("realizedPnl") or 0) > 0 else 0})
h = pd.DataFrame(rows)
h["res_ts"] = (pd.to_datetime(h["endDate"], errors="coerce", utc=True).astype("int64") // 10**9)
h = h[(h["entry_ts"] > 0) & (h["res_ts"] > 0) & (h["res_ts"] > h["entry_ts"]) & (h["entry"] > 0)].copy()
h = h.sort_values("entry_ts").reset_index(drop=True)
qmask = np.zeros(len(h), dtype=bool)
for wallet, idx in h.groupby("wallet").groups.items():
    w = h.loc[idx]; rs = w.sort_values("res_ts")
    rt = rs["res_ts"].values; wc = np.cumsum(rs["won"].values)
    k = np.searchsorted(rt, w["entry_ts"].values, side="left")
    wins = np.where(k > 0, wc[np.clip(k - 1, 0, len(wc) - 1)], 0)
    wr = np.where(k > 0, wins / np.maximum(k, 1), 0.0)
    qmask[np.array(idx)] = (k >= CFG.WF_MIN_TRAILING_TRADES) & (wr >= CFG.WF_MIN_TRAILING_WINRATE)
q = h[qmask].copy()
q["backers"] = q.groupby(["conditionId", "outcome"])["wallet"].transform("nunique")
print(f"{len(q)} PIT-qualified positions")

4175 PIT-qualified positions


## 2. Reconstruct each sharp's exits from their trade activity
We pull each involved wallet's full BUY/SELL history once, then per (wallet, token) find how
much they sold and *when they last sold*. That last-sell time is our exit trigger; if they
barely sold, we treat it as held to resolution.

In [3]:
sample = q.sample(n=min(CFG.MIRROR_SAMPLE, len(q)), random_state=42).copy()
wallets = sample["wallet"].unique()
print(f"pulling trade history for {len(wallets)} wallets (cached after first run)...")

# (wallet, asset) -> {buy_size, sell_size, last_sell_ts}
flow = {}
for j, wal in enumerate(wallets):
    for t in get_user_trades(wal):
        key = (wal, t.get("asset"))
        f = flow.setdefault(key, {"buy": 0.0, "sell": 0.0, "last_sell_ts": 0})
        sz = float(t.get("size") or 0)
        if t.get("side") == "BUY":
            f["buy"] += sz
        elif t.get("side") == "SELL":
            f["sell"] += sz
            f["last_sell_ts"] = max(f["last_sell_ts"], int(t.get("timestamp") or 0))
    if (j + 1) % 15 == 0:
        print(f"  {j+1}/{len(wallets)} wallets")
print(f"reconstructed flow for {len(flow)} (wallet, token) positions")

pulling trade history for 53 wallets (cached after first run)...
[warn] GET failed after retries: https://data-api.polymarket.com/activity params={'user': '0x6db568e61e5e3de7d87f831431b673f38ce2e279', 'type': 'TRADE', 'start': 0, 'sortBy': 'TIMESTAMP', 'sortDirection': 'ASC', 'limit': 500, 'offset': 3500} err=400 Client Error: Bad Request for url: https://data-api.polymarket.com/activity?user=0x6db568e61e5e3de7d87f831431b673f38ce2e279&type=TRADE&start=0&sortBy=TIMESTAMP&sortDirection=ASC&limit=500&offset=3500
[warn] GET failed after retries: https://data-api.polymarket.com/activity params={'user': '0x9f2fe025f84839ca81dd8e0338892605702d2ca8', 'type': 'TRADE', 'start': 0, 'sortBy': 'TIMESTAMP', 'sortDirection': 'ASC', 'limit': 500, 'offset': 3500} err=400 Client Error: Bad Request for url: https://data-api.polymarket.com/activity?user=0x9f2fe025f84839ca81dd8e0338892605702d2ca8&type=TRADE&start=0&sortBy=TIMESTAMP&sortDirection=ASC&limit=500&offset=3500
[warn] GET failed after retries: ht

## 3. Simulate the lagged copy
Enter at the price `COPY_LAG_HOURS` after their buy; exit at the price `COPY_LAG_HOURS`
after their last sell (if they sold ≥ half the position) — otherwise hold to resolution.
Slippage is paid on both legs.

In [4]:
LAG = CFG.COPY_LAG_HOURS * 3600
S = CFG.SLIPPAGE_TOLERANCE

def simulate_row(r):
    asset = r["asset"]
    buy = price_at(asset, r["entry_ts"] + LAG)
    if buy is None:
        return None
    buy = buy * (1 + S)                                   # we pay up entering late
    if not (CFG.PRICE_FLOOR <= buy <= CFG.PRICE_CEILING):
        return None
    f = flow.get((r["wallet"], asset), {"buy": 0.0, "sell": 0.0, "last_sell_ts": 0})
    sold_enough = f["sell"] >= 0.5 * max(f["buy"], 1e-9) and f["last_sell_ts"] > r["entry_ts"]
    if sold_enough:
        sell = price_at(asset, f["last_sell_ts"] + LAG)
        if sell is None:
            sell = r["won"]                                # fallback: resolution value
        else:
            sell = sell * (1 - S)                          # we receive less exiting late
    else:
        sell = float(r["won"])                            # held to resolution (1 if won else 0)
    return (sell - buy) / buy                             # copier's return on the position

sample["copy_ret"] = sample.apply(simulate_row, axis=1)
sim = sample.dropna(subset=["copy_ret"])
print(f"{len(sim)} positions simulated (rest skipped: no price data or out of band)")

104 positions simulated (rest skipped: no price data or out of band)


## 4. Verdict — lagged mirror vs the frictionless ceiling

In [5]:
def summ(df, n):
    s = df[df["backers"] >= n]
    if s.empty:
        return None
    return {"min_backers": n, "n": len(s),
            "lagged_mean": round(s["copy_ret"].mean(), 3),
            "lagged_median": round(s["copy_ret"].median(), 3),
            "%_profitable": round((s["copy_ret"] > 0).mean(), 3)}

res = pd.DataFrame([r for r in (summ(sim, n) for n in [1, 2, 3]) if r])
print(f"LAGGED MIRROR ({CFG.COPY_LAG_HOURS}h delay, slippage both legs), out-of-sample:")
print("(compare to nb08 ceiling: mirror_median +0.057 / +0.163 / +0.199)")
res

LAGGED MIRROR (6h delay, slippage both legs), out-of-sample:
(compare to nb08 ceiling: mirror_median +0.057 / +0.163 / +0.199)


,min_backers,n,lagged_mean,lagged_median,%_profitable
0,1,104,0.374,0.183,0.625
1,2,33,0.551,0.158,0.727
2,3,13,0.582,0.130,0.615


## 5. The decision
- **`lagged_median` clearly > 0 on a healthy `n`** → the exit edge survives realistic delay.
  This is a genuine, tradeable copy strategy. Next steps: re-check across `COPY_LAG_HOURS`
  (1h / 6h / 24h) for robustness, then **paper-trade the live mirror** for 4–6 weeks before
  any capital, and only then consider gated execution (Blueprint §11).
- **`lagged_median` ≈ 0 or negative** → the ceiling was real but only reachable with zero
  latency; being a step behind erases it. That is the wall, and it's an honest, well-earned
  place to stop.

### Read it carefully
- Try a few values of `CFG.COPY_LAG_HOURS`. If the edge only exists at 1h but dies by 6–24h,
  it demands near-real-time infrastructure, which changes the project entirely.
- Survivorship (today's leaderboard) and equal-weighting still apply. And this is a *sample*
  of positions — raise `CFG.MIRROR_SAMPLE` to tighten the estimate once you see it work.
- The exit rule ("sold ≥ half → exit at last sell") is a reasonable approximation of messy,
  partial real-world selling; a positive result would justify modelling it more precisely.